In [123]:
import os
import glob
from google import genai
import re
import shutil
import unicodedata
import time
from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import user, system
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter, SpacyTextSplitter
import pdfplumber
import json

In [43]:
MODEL = "grok-4.20-0309-reasoning"
GEMINI_MODEL = "gemini-3.1-flash-lite"
book_ratio = 50 #What % of the book has to be left

In [44]:
def get_paths():
    pdf_list = glob.glob("input/*.pdf")
    pdf_list = [re.sub(r'\\', '/', pdf) for pdf in pdf_list]
    print(pdf_list[0])
    return pdf_list

In [45]:
def extract_text(pdf_path):
    """Extract all text from a PDF. Returns (full_text, page_count)."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        page_count = len(pdf.pages)
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n\n".join(pages), page_count

In [104]:
def chunk_book(text):
    chunker = RecursiveCharacterTextSplitter(chunk_size=6000, chunk_overlap=200)
    chunks = chunker.split_text(text)
    print(f"Chunks: {len(chunks)}")
    return chunks

In [47]:
def convert_pdfs_to_ascii():
    pdf_list = glob.glob("input/*.pdf")
    pdf_list = [re.sub(r'\\', '/', pdf) for pdf in pdf_list]

    for pdf in pdf_list:
        mod_pdf = pdf.replace("input", "ascii_input")
        if os.path.exists(mod_pdf):
            print(f"{mod_pdf} already exists")
        else:
            base = os.path.basename(pdf)
            ascii_base = unicodedata.normalize("NFKD", base).encode("ascii", "ignore").decode("ascii")
            ascii_base = re.sub(r"[^A-Za-z0-9._-]+", "_", ascii_base)
            safe_path = os.path.join("ascii_input", ascii_base)
            # Keep original file; just make a copy with safe name
            shutil.copy2(pdf, safe_path)
            print(f"{mod_pdf} created")
    return None

In [48]:
def upload_pdfs_to_gemini():
    client = genai.Client()
    pdfs=glob.glob("ascii_input/*.pdf")
    file_pdf_list = []

    # Upload each PDF
    for pdf in pdfs:
        myfile = client.files.upload(file=pdf)
        file_name = myfile.name
        myfile = client.files.get(name=file_name)
        print(myfile)
        file_pdf_list.append({"book_name": os.path.basename(pdf), "file": myfile})

        # Wait for the file to be active
        while myfile.state.name != "ACTIVE":
            time.sleep(1)
            myfile = client.files.get(name=file_name)
            print(myfile)

    return file_pdf_list

In [49]:
def upload_pdfs_to_grok():
    pdfs=glob.glob("ascii_input/*.pdf")
    grok_file_pdf_list = []

    # Upload each PDF
    load_dotenv()
    client = Client(api_key=os.getenv("XAI_API_KEY"))
    for pdf in pdfs:
        file = client.files.upload(pdf, expires_after=82800) #expires after 23 hours
        grok_file_pdf_list.append({"book_name": os.path.basename(pdf), "file": file})



    return grok_file_pdf_list

In [50]:
load_dotenv()
client = Client(api_key=os.getenv("XAI_API_KEY"))

In [51]:
GROK_SYSTEM = """\
Tu esi lietuvių literatūros teksto atkūrimo redaktorius.

<constraints>
MUST: Naudok tik originalaus failo ir Gemini versijos turinį.
MUST: Galutinio teksto tobulas ilgis — ±{target_len} simbolių.
MUST: Leistinas teksto diapazonas: {lo_len}–{hi_len} simbolių.
MUST: Išlaik autoriaus sakinių ritmą, leksiką ir pasakojimo toną.
MUST: Atkurk viską, kas prarasta — vardus, vietas, datas, dialogo fragmentus, emocines reakcijas, aplinkos detales.
NEVER: Nepridėk informacijos, kurios nėra originale.
NEVER: Grąžink tik tekstą — jokių komentarų, įžangų, paaiškinimų.
NEVER: Nerašyk AI stiliumi — tekstas turi skambėti kaip autorius.
</constraints>

<task>
Vienas praėjimas: surask, ko trūksta Gemini trumpintoje versijoje lyginant su originalu, įterpk į Gemini struktūrą, ištaisyk nenatūralią lietuvių kalbą. Jei reikia jungties su ankstesniu fragmentu — maksimaliai 1–2 frazės iš originalo turinio. Grąžink tik rezultatą.
</task>

<reasoning_discipline>
NEVER: Necituok ir nerašyk pilno teksto reasoning metu.
MUST: Reasoning — tik trūkstamų elementų sąrašas trumpais įrašais (pvz.: "vardas X → įterpti 3 par.", "dialogas → atkurti"). Jokio teksto perrašymo.
MUST: Kuo mažiau reasoning — kuo daugiau tiesiogiai į rezultatą.
</reasoning_discipline>
"""

GROK_USER = """\
<original_file.id>
{file}
</original_file.id>

<gemini_compressed_version>
{compressed}
</gemini_compressed_version>

Atkurk ir patobulink sutrumpintą fragmentą remdamasis originalu. Pateik tik galutinį tekstą.
Galutinis tekstas:\
"""

In [52]:
GEMINI_SYSTEM = """\
Tu esi aukštos kvalifikacijos lietuvių literatūros redaktorius ir teksto trumpintojas.
Tavo užduotis yra sutrumpinti pateiktą knygos fragmentą TIKSLIAI iki {ratio_pct}% jo originalaus simbolių skaičiaus.

SVARBIOS TAISYKLĖS:
1. ILGIO REIKALAVIMAS — PRIVALOMAS: rezultato tekstas turi būti {ratio_pct}% originalaus fragmento ilgio (±5%).
2. Išlaik VISUS esminius siužeto įvykius, pagrindinių veikėjų vystymąsi ir svarbias scenas.
3. Išlaik autoriaus kalbos stilių ir toną kiek įmanoma.
4. Sumažink antrinius aprašymus, pasikartojančias mintis ir per ilgus dialogus.
5. Trumpink, bet NEKURK naujų faktų ar įvykių — tik rinktinai šalink.
6. Rezultatas turi būti sklandžiai skaitomas lietuviškas tekstas.
7. NEANALIZUOK ir NESKAIČIUOK — tiesiog pateik sutrumpintą tekstą be jokių komentarų, skaičiavimų ar įžangų.
8. Nepridėk jokių antraščių ar paaiškinimų — pradėk tiesiogiai nuo teksto.\
"""

GEMINI_USER = """\
Esi teksto redaktorius. Sutrumpink pateiktą knygos fragmentą.

REIKALAVIMAI:
- Tikslinis ilgis: {target_len} simbolių
- Leistinas diapazonas: {lo_len}–{hi_len} simbolių ({ratio_pct}% originalo)
- PRIVALOMA: Prieš rašydamas patikrink savo teksto ilgį. Jei per trumpas — plėsk. Jei per ilgas — trumpink. Kartok kol patenki į diapazoną.
- Išlaikyk originalo stilių, toną ir svarbias detales
- Nekomentuok, nerašyk skaičių ar paaiškinimų

PROCESAS:
1. Apskaičiuok: {target_len} simbolių = maždaug {target_len} / 5 = ~{target_len} žodžių
2. Rašyk sutrumpintą tekstą
3. Įvertink: ar ilgis tarp {lo_len}–{hi_len}? Jei ne — koreguok

--- FRAGMENTAS ---
{chunk}
--- FRAGMENTO PABAIGA ---

Sutrumpintas tekstas (TARP {lo_len} IR {hi_len} SIMBOLIŲ):\
"""
GEMINI_USER_OLD = """\
Sutrumpink toliau pateiktą knygos fragmentą iki {ratio_pct}% jo dydžio.

Tikslinis ilgis: {target_len} simbolių (leistinas diapazonas: {lo_len}–{hi_len}).

--- ORGINALAUS TEKSTO FRAGMENTAS ---
{chunk}
--- ORGINALAUS TEKSTO FRAGMENTO PABAIGA ---

Pateik tik sutrumpintą fragmento tekstą. Jokių komentarų ar skaičiavimų.
Galutinis tekstas:\
"""

In [53]:
def make_grok_prompt(file_id, gemini_chunk, target_len):
    lo = target_len * .95
    hi = target_len * 1.05

    user_prompt = user(GROK_USER.format(file=file_id, compressed=gemini_chunk))
    system_prompt = system(GROK_SYSTEM.format(target_len=target_len, lo_len=lo, hi_len=hi))

    return system_prompt, user_prompt

In [54]:
def make_batch(model, gemini_chunk, file_id, batch_id, request_index, target_len):
    system_prompt, user_prompt = make_grok_prompt(file_id, gemini_chunk, target_len)

    chat = client.chat.create(
        model=model,
        batch_request_id = f"{batch_id}-{request_index}"
    )
    chat.append(system_prompt)
    chat.append(user_prompt)

    return chat

In [122]:
def make_request(book_name, file_id, gemini_chunks, chunks):
    batch = client.batch.create(batch_name=book_name)
    batch_id = batch.batch_id
    batch_requests = []

    for i, chunk in enumerate(start=0, iterable=gemini_chunks):
        chunkBatch = make_batch(MODEL, chunk, file_id, batch_id, request_index=i+1, target_len=int(len(chunks[i]) * book_ratio / 100))
        batch_requests.append(chunkBatch)

    print(f"Requests: {len(batch_requests)}")
    client.batch.add(batch_id=batch_id, batch_requests=batch_requests)
    return batch_id

In [57]:
def convert_books_gemini():
    batch_list = []

    print("uploading files to files api")
    geminiFiles = upload_pdfs_to_gemini()
    text, pageCount = extract_text(pdf)
    chunks = chunk_book(text)

    #Gemini shortening
    for file in geminiFiles:
        print(f"Trumpinamas failas: {file}")
        batch_list.append(call_gemini_batch(chunks, file))

    return batch_list

In [121]:
load_dotenv(override=True)
gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def gemini_call(chunk, num):
    target = len(chunk) * book_ratio / 100
    lo = target*0.9
    hi=target*1.1
    system = GEMINI_SYSTEM.format(ratio_pct=book_ratio)
    user = GEMINI_USER.format(chunk=chunk, ratio_pct=book_ratio, target_len=target, lo_len=lo, hi_len=hi)
    print(f"Gemini working on chunk: {num}")

    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            {"role": "user", "parts": [{"text": user}]}
        ],
        config={"system_instruction": system}
    )
    return response




In [119]:
def shorten_gemini():
    files = glob.glob("ascii_input/*.pdf")
    gemini_books = []
    book_chunks = []

    for file in files:
        converted = ""
        chunk_list = []

        text, pageCount = extract_text(file)
        chunks = chunk_book(text)

        for i, chunk in enumerate(start=1, iterable=chunks):
            response = gemini_call(chunk, i)
            converted += "\n" + response.text
            chunk_list.append(chunk)

        gemini_books.append(converted)
        book_chunks.append(chunk_list)

    return gemini_books, book_chunks

In [60]:
def convert_books_grok():
    grokFiles = upload_pdfs_to_grok()
    gemini_books = shorten_gemini()
    gemini_chunks = []

    for i in range(len(gemini_books)):
        gemini_chunks.append(chunk_book(gemini_books[i]))

    batchIds = []

    for i, file in enumerate(grokFiles):
        bookName = file["book_name"]
        file_id = file["file"].name
        batchIds.append(make_request(file_id, gemini_chunks[i]["content"]))

    return batchIds

In [97]:
convert_pdfs_to_ascii()

ascii_input/Ad_astra.pdf created
ascii_input/Jonas_Biliūnas._tik_Liūdna_pasaka.LG1800.pdf created


In [103]:
gemini_books, book_chunks = shorten_gemini()

Chunks: 2
Chunk len: 5869
Target len: 2934.5
Chunk len: 4646
Target len: 2323.0
Chunks: 12
Chunk len: 3565
Target len: 1782.5
Chunk len: 3838
Target len: 1919.0
Chunk len: 4036
Target len: 2018.0
Chunk len: 3865
Target len: 1932.5
Chunk len: 4301
Target len: 2150.5
Chunk len: 3772
Target len: 1886.0
Chunk len: 3661
Target len: 1830.5
Chunk len: 3776
Target len: 1888.0
Chunk len: 3902
Target len: 1951.0
Chunk len: 3659
Target len: 1829.5
Chunk len: 3775
Target len: 1887.5
Chunk len: 4440
Target len: 2220.0


In [106]:
grokFiles = upload_pdfs_to_grok()

In [ ]:
gemini_chunks = []

for i in range(len(gemini_books)):
    gemini_chunks.append(chunk_book(gemini_books[i]))

0
Chunks: 1
1
Chunks: 5


In [108]:
batchIds = []

for i, file in enumerate(grokFiles):
    bookName = file["book_name"]
    file_id = file["file"].id
    batchIds.append(make_request(bookName, file_id, gemini_chunks[i], book_chunks[i]))

Creating batch: 0
Requests: 1
Creating batch: 0
Creating batch: 1
Creating batch: 2
Creating batch: 3
Creating batch: 4
Requests: 5


In [111]:
#Check batch list
response = client.batch.list(limit=20)
for batch in response.batches:
    status = "complete" if batch.state.num_pending == 0 else "processing"
    print(f"{batch.name} ({batch.batch_id}): {status}")

Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_eab4c5d5-ca4b-4a7a-8bf1-888883378501): complete
Ad_astra.pdf (batch_2fc19623-f4af-4226-a58c-43b2ad0fbbbf): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_0b53f54c-6e4d-484c-ba3e-7a15d05d797c): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_836f486e-be03-492d-854d-b29e5ef0b4a0): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_b2c8c6c0-a1a3-4b9a-8db4-b2e217346266): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_739167ce-297b-4161-920d-ecaf5cc4126b): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_b177ef39-1fb1-4670-bb9a-10d226126d26): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_7470f6c2-b1b8-440c-84a5-34f2f85f801d): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_91f90d27-a310-4e67-9e7f-975e7fa306bd): complete
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_2ba0a24b-dbb0-4971-b11d-362d6fca8ef6): complete


In [110]:
print(batchIds)

['batch_2fc19623-f4af-4226-a58c-43b2ad0fbbbf', 'batch_eab4c5d5-ca4b-4a7a-8bf1-888883378501']


In [112]:
def wait_for_grok_batch(batchID):
    load_dotenv()
    client = Client(api_key=os.getenv("XAI_API_KEY"))
    # Paginate through all results
    all_succeeded = []
    all_failed = []
    pagination_token = None
    while True:
        # Fetch a page of results (limit controls page size)
        page = client.batch.list_batch_results(
            batch_id=batchID,
            limit=100,
            pagination_token=pagination_token,
        )
        
        # Collect results from this page
        all_succeeded.extend(page.succeeded)
        all_failed.extend(page.failed)
        
        # Check if there are more pages
        if page.pagination_token is None:
            break
        pagination_token = page.pagination_token
    # Process results - handle different response types
    print(f"Successfully processed: {len(all_succeeded)} requests")
    for result in all_succeeded:
        rid = result.batch_request_id
        resp = result.proto.response
        if resp.HasField("completion_response"):
            # Chat completion response
            print(f"[{rid}] {result.response.content}")
            print(f"  Tokens used: {result.response.usage.total_tokens}")
    if all_failed:
        print(f"\nFailed: {len(all_failed)} requests")
        for result in all_failed:
            print(f"[{result.batch_request_id}] Error: {result.error_message}")

    return all_succeeded, all_failed

In [58]:
#Delete files
client = genai.Client()

print('My files:')
for f in client.files.list():
    print(' ', f.name)
    client.files.delete(name=f.name)

My files:


In [ ]:
#Check batches
client = genai.Client()

print('My files:')
for f in client.batches.list():
    print(f"ID: {f.name}, State: {f.state}")
    

My files:


In [ ]:
response = client.files.list(
    limit=10,
    order="desc",
    sort_by="created_at"
)
for file in response.data:
    expires = file.expires_at.ToDatetime() if file.HasField("expires_at") else "never"
    print(f"File: {file.filename} (ID: {file.id}, Size: {file.size} bytes, Expires: {expires})")

File: Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (ID: file_92b30b2d-8dfa-4384-a666-2f5f3144adf2, Size: 230204 bytes, Expires: 2026-05-12 14:02:57.350721)


TypeError: bad argument type for built-in operation

In [99]:
cancelled_batch = client.batch.cancel(batch_id='batch_2ba0a24b-dbb0-4971-b11d-362d6fca8ef6')
print(f"Cancelled batch: {cancelled_batch.batch_id}")
print(f"Completed before cancellation: {cancelled_batch.state.num_success} requests")

Cancelled batch: batch_2ba0a24b-dbb0-4971-b11d-362d6fca8ef6
Completed before cancellation: 0 requests


In [114]:
print(succeeded)

[batch_request_id: "batch_eab4c5d5-ca4b-4a7a-8bf1-888883378501-1"
response {
  completion_response {
    id: "594f5ba6-f223-9b3f-aec0-ffe493bccfb1"
    outputs {
      finish_reason: REASON_STOP
      message {
        content: "O jaunosios dienos mano! Kaip melsvam ore gervės nykstate jūs, tiktai ką pasirodžiusios... Žiūriu į jus, pralėkusias, kaip į sapną gražų, ir matau, kad jau artinas ruduo gyvenimo mano. O Ramūta! Kodėl nepažinau tavęs, dar mažutis, nekaltas būdamas! Tada tokios gražios, laimingos buvo dienos. Būtume Šventosios pakrančiais vaikščioję, lakštingalos giesmių klausę. Bet tu buvai toli nuo manęs, ir aš net nežinojau, kad tu gyveni pasaulyje... Taip, nežinojau, bet jaučiau... Nuo pat mažų dienų tavo paveikslas mano širdyje gyveno. Kai gimnazijon važiuodavau ir tavo tėvynės klonius pro langą išvysdavau, krūtinėje sujudėdavo naujas jautimas: tarsi paukštis pro šalį pralėkdavo, bet vis žiūrėdavau į išsitiesusį paveikslą, kolei jis išnykdavo... Tik tau vienai nebijau savo 

In [129]:
print(grokFiles)

[{'book_name': 'Ad_astra.pdf', 'file': size: 90342
created_at {
  seconds: 1778517185
  nanos: 199579000
}
expires_at {
  seconds: 1778599985
  nanos: 195443000
}
filename: "Ad_astra.pdf"
id: "file_0ed2b9f1-1dd5-4d85-ad6f-2cb66fa08506"
}, {'book_name': 'Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf', 'file': size: 230204
created_at {
  seconds: 1778517186
  nanos: 153992000
}
expires_at {
  seconds: 1778599986
  nanos: 149888000
}
filename: "Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf"
id: "file_08aa99b4-1fc1-4d2a-a5b2-fb40b852f333"
}]


In [136]:

batchInfo = [{"book_name": file["book_name"], "batchId": b_id} for file, b_id in zip(grokFiles, batchIds)]
batchInfo
#json.dump(batchInfo, open("batch_ids.json", "w"))

[{'book_name': 'Ad_astra.pdf',
  'batchId': 'batch_2fc19623-f4af-4226-a58c-43b2ad0fbbbf'},
 {'book_name': 'Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf',
  'batchId': 'batch_eab4c5d5-ca4b-4a7a-8bf1-888883378501'}]

In [117]:
def procces_final_files(all_succeeded, all_failed):
    book_chunks = []
    for result in all_succeeded:
        if result.proto.response.HasField("completion_response"):
            book_chunks.append(result.response.content)
    if all_failed:
        for result in all_failed:
            print(f"[{result.batch_request_id}] Error: {result.error_message}")
    return book_chunks

In [116]:
def store_all_books():
    knygos = []
    for batchId, file in zip(batchIds, grokFiles):
        succeeded, failed = wait_for_grok_batch(batchId)
        knygos.append(procces_final_files(succeeded, failed))

    for i, file in enumerate(grokFiles):
        name = os.path.splitext(file["book_name"])[0]
        with open(f"Final_output/{name}.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(knygos[i]))

In [118]:
store_all_books()

Successfully processed: 1 requests
[batch_2fc19623-f4af-4226-a58c-43b2ad0fbbbf-1] Ūkininkas Dalba šventadienio rytą jautėsi neįprastai pakylėtas. Jis siekė reformuoti savo ūkį: išnaikino senas trobas, išrovė vyšnias, o dabar nusprendė atsikratyti ir nereikalingu tapusio seno šuns. Nors žmona įspėjo nedaryti nieko pikta šventą dieną, Dalba, sočiai papusryčiavęs riebių blynų su spirgučiais, leidosi upės ledu link miestelio, sekamas sukvaišusio, veršio dydžio šuns, vardu Ciucka.

Žiemos dangus buvo niūrus, protarpiais nušvintantis tolimos saulės atšvaitais. Ant upės ledo vaikai džiaugėsi rogutėmis, o šunys triukšmingai šėlo. Senas šuo, atstumtas kitų gentainių, nuleidęs galvą sekė šeimininką. Aukštai ant kranto, vieškeliu, dundėjo vežimai, o prabangiausiose rogėse sėdėjo skunkso kailiniais pasidabinusi ponia.

Ties vieta, kur vasarą būdavo brasta, o dabar tyvuliavo tik skalbimui skirta aketė, Dalba nutarė įgyvendinti savo sumanymą. Jis saldžiu balsu pasikvietė šunį. Šuo, įpratęs pasitikėt

In [139]:
print(batchIds)

['batch_2fc19623-f4af-4226-a58c-43b2ad0fbbbf', 'batch_eab4c5d5-ca4b-4a7a-8bf1-888883378501']


In [143]:
def procces_final_files(all_succeeded, all_failed):
    book_chunks = []
    for result in all_succeeded:
        if result.proto.response.HasField("completion_response"):
            book_chunks.append(result.response.content)
    if all_failed:
        for result in all_failed:
            print(f"[{result.batch_request_id}] Error: {result.error_message}")
    return book_chunks

def store_all_books():
    knygos = []
    batchInfo = json.load(open("batch_ids.json"))

    for batch in (batchInfo):
        batchId = batch["batchId"]
        succeeded, failed = wait_for_grok_batch(batchId)
        knygos.append(procces_final_files(succeeded, failed))

    for i, id in enumerate(batchInfo):
        name = id["book_name"]
        with open(f"Final_output/{name}.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(knygos[i]))

In [145]:
store_all_books()

Successfully processed: 1 requests
[batch_2fc19623-f4af-4226-a58c-43b2ad0fbbbf-1] Ūkininkas Dalba šventadienio rytą jautėsi neįprastai pakylėtas. Jis siekė reformuoti savo ūkį: išnaikino senas trobas, išrovė vyšnias, o dabar nusprendė atsikratyti ir nereikalingu tapusio seno šuns. Nors žmona įspėjo nedaryti nieko pikta šventą dieną, Dalba, sočiai papusryčiavęs riebių blynų su spirgučiais, leidosi upės ledu link miestelio, sekamas sukvaišusio, veršio dydžio šuns, vardu Ciucka.

Žiemos dangus buvo niūrus, protarpiais nušvintantis tolimos saulės atšvaitais. Ant upės ledo vaikai džiaugėsi rogutėmis, o šunys triukšmingai šėlo. Senas šuo, atstumtas kitų gentainių, nuleidęs galvą sekė šeimininką. Aukštai ant kranto, vieškeliu, dundėjo vežimai, o prabangiausiose rogėse sėdėjo skunkso kailiniais pasidabinusi ponia.

Ties vieta, kur vasarą būdavo brasta, o dabar tyvuliavo tik skalbimui skirta aketė, Dalba nutarė įgyvendinti savo sumanymą. Jis saldžiu balsu pasikvietė šunį. Šuo, įpratęs pasitikėt